In [0]:
%run ../Databricks-Bookstore-Engineering/Copy-Datasets

In [0]:


current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
dataset_bookstore = f"/Volumes/{current_catalog}/bookstore_eng_pro/dataset"
checkpoint_path = f"/Volumes/{current_catalog}/bookstore_eng_pro/checkpoints"
print(dataset_bookstore)

In [0]:
%sql

SELECT cast(key as STRING), cast(value as STRING)
from bronze 
where topic = "orders"
limit 20

In [0]:
%sql
select * from bronze where topic = "orders"

In [0]:
%sql 

SELECT v.*
from (
   select from_json(cast(value as STRING), "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>") v
   from bronze
   where topic = "orders"
)


In [0]:
(spark.readStream
 .table("bronze")
 .createOrReplaceTempView("bronze_tmp"))


In [0]:
spark.conf.set(
    "spark.sql.streaming.checkpointLocation",
    "/Volumes/certspace/bookstore_eng_pro/dataset/checkpoints/interactive_sql"
)

In [0]:
%sql 

SELECT v.*
from (
   select from_json(cast(value as STRING), "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>") v
   from bronze_tmp
   where topic = "orders"
)


In [0]:
%sql

CREATE OR REPLACE TEMPORARY VIEW orders_silver_tmp as SELECT v.*
from (
   select from_json(cast(value as STRING), "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>") v
   from bronze_tmp
   where topic = "orders"
)


In [0]:
query = (spark.table("orders_silver_tmp")
         .writeStream
         .option("checkpointLocation", f"{checkpoint_path}/orders_silver")
         .trigger(availableNow=True)
         .toTable("orders_silver"))

query.awaitTermination()

In [0]:
%sql
select * from orders_silver_tmp

In [0]:
%sql
drop table orders_silver

In [0]:
existing_query = next((q for q in spark.streams.active if q.name == "orders_silver"), None)
if existing_query is not None:
    existing_query.stop()
    existing_query.awaitTermination()

In [0]:
from pyspark.sql import functions as F

json_schema = "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>"

query = (spark.readStream.table("bronze")
        .filter("topic = 'orders'")
        .select(F.from_json(F.col("value").cast("string"), json_schema).alias("v"))
        .select("v.*")
     .writeStream
        .queryName("orders_silver")
        .option("checkpointLocation", f"/Volumes/certspace/bookstore_eng_pro/dataset/checkpoints/orders_silver1")
        .trigger(availableNow=True)
        .table("orders_silver"))

query.awaitTermination()

In [0]:
%sql 
select * from orders_silver